# 차원 이름 끝 공백 진단 (KPI_W / ktws)

`DIM_MNG_USER` 등의 이름 컬럼에 눈에 보이지 않는 **끝 공백**이 들어간 값이 있다.
합계에는 포함되지만, 그 값으로 **필터를 걸면 결과가 0건**이 된다.

원인은 GOLD 쿼리가 쓰는 LIKE 패딩 필터다.

```sql
@GroupNamePad LIKE N'%,' + U.group_name + N',%'
```

`group_name`이 `'토요타 동대문 '`(끝 공백)이면 패턴이 `'%,토요타 동대문 ,%'`가 되는데,
사용자가 넘긴 값은 `',토요타 동대문,'`이라 **매칭이 실패**한다.
LIKE는 패턴의 끝 공백을 무시하지 않기 때문이다.

## 주의: `=` 비교로는 못 찾는다

SQL Server는 문자열 비교에서 끝 공백을 무시한다 — `'abc ' = 'abc'` 가 **참**이다.
그래서 `WHERE col <> LTRIM(RTRIM(col))` 로 찾으면 **아무것도 안 나온다**.
반드시 `DATALENGTH` 로 비교해야 한다. (이 노트북이 그 대조도 함께 보여준다)

발견 경위: 2026-08-03, 퍼널 전시장 grain 검증 중 `토요타 동대문`이 매칭되지 않아 확인.


## 0. 준비

In [ ]:
# 필요시에만 실행
# %pip install pyodbc pandas python-dotenv

## 1. 접속

자격 증명은 노트북에 적지 않는다 — 대시보드와 같은 `.env`(`Fabric_ID` / `Fabric_PW`)에서 읽는다.
`.env`가 없으면 `ActiveDirectoryInteractive`(브라우저 로그인)로 넘어간다.

In [ ]:
import os
import pyodbc
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# 이 노트북 위치에서 대시보드 .env 를 찾는다. 경로가 다르면 ENV_PATH 를 직접 지정할 것.
ENV_PATH = Path.cwd()
for _ in range(5):
    if (ENV_PATH / "main" / "dashboard" / ".env").exists():
        ENV_PATH = ENV_PATH / "main" / "dashboard" / ".env"
        break
    if (ENV_PATH / ".env").exists() and (ENV_PATH / "server" / "fabricClient.js").exists():
        ENV_PATH = ENV_PATH / ".env"
        break
    ENV_PATH = ENV_PATH.parent
load_dotenv(ENV_PATH if ENV_PATH.is_file() else None)

# KPI_W 는 BP_KTWS 엔드포인트에 있다 (server/fabricClient.js 의 DB_TO_SYSTEM 참고)
SERVER = "REPLACE_ME.datawarehouse.fabric.microsoft.com"
DATABASE = "KPI_W"

uid, pwd = os.getenv("Fabric_ID"), os.getenv("Fabric_PW")
if uid and pwd:
    auth = f"Authentication=ActiveDirectoryPassword;UID={uid};PWD={pwd};"
    print(f"자격 증명 사용: {uid}")
else:
    auth = "Authentication=ActiveDirectoryInteractive;"
    print("Fabric_ID/PW 없음 → 브라우저 로그인으로 진행")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER},1433;DATABASE={DATABASE};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;" + auth
)
pd.read_sql("SELECT DB_NAME() AS db, SYSDATETIME() AS now", conn)

## 2. `=` 비교는 왜 못 찾는가

아래 두 쿼리는 **같은 컬럼**을 보지만 결과가 다르다.
`=`/`<>` 는 끝 공백을 무시하므로 0건이 나오고, `DATALENGTH` 로 재야 실제 값이 잡힌다.

In [ ]:
sql_wrong = """
SELECT COUNT(*) AS 찾은_행수
FROM ktws.DIM_MNG_USER
WHERE group_name <> LTRIM(RTRIM(group_name))
"""
sql_right = """
SELECT COUNT(*) AS 찾은_행수
FROM ktws.DIM_MNG_USER
WHERE DATALENGTH(group_name) <> DATALENGTH(LTRIM(RTRIM(group_name)))
"""
print("<> 비교 (끝 공백 무시됨):", pd.read_sql(sql_wrong, conn).iloc[0, 0])
print("DATALENGTH 비교 (정확)  :", pd.read_sql(sql_right, conn).iloc[0, 0])

## 3. 어느 컬럼에 몇 개나 있는가

In [ ]:
sql = """
SELECT 컬럼, COUNT(*) AS 값_종류, SUM(행수) AS 영향_행수
FROM (
    SELECT 'DIM_MNG_USER.group_name (전시장)' AS 컬럼, group_name AS v, COUNT(*) AS 행수
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(group_name) <> DATALENGTH(LTRIM(RTRIM(group_name)))
     GROUP BY group_name
    UNION ALL
    SELECT 'DIM_MNG_USER.dept_nm (팀)', dept_nm, COUNT(*)
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(dept_nm) <> DATALENGTH(LTRIM(RTRIM(dept_nm)))
     GROUP BY dept_nm
    UNION ALL
    SELECT 'DIM_MNG_USER.name (SC)', name, COUNT(*)
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(name) <> DATALENGTH(LTRIM(RTRIM(name)))
     GROUP BY name
    UNION ALL
    SELECT 'DIM_MNG_DEALER.dealer_nm (딜러)', dealer_nm, COUNT(*)
      FROM ktws.DIM_MNG_DEALER
     WHERE DATALENGTH(dealer_nm) <> DATALENGTH(LTRIM(RTRIM(dealer_nm)))
     GROUP BY dealer_nm
) t
GROUP BY 컬럼
ORDER BY 영향_행수 DESC
"""
pd.read_sql(sql, conn)

## 4. 실제로 어떤 값들인가

대괄호로 감싸 공백이 눈에 보이게 한다.

In [ ]:
sql = """
SELECT '전시장' AS 구분, '[' + group_name + ']' AS 값,
       DATALENGTH(group_name) AS 바이트,
       DATALENGTH(LTRIM(RTRIM(group_name))) AS trim_바이트,
       COUNT(*) AS 사용자수
  FROM ktws.DIM_MNG_USER
 WHERE DATALENGTH(group_name) <> DATALENGTH(LTRIM(RTRIM(group_name)))
 GROUP BY group_name
UNION ALL
SELECT '팀', '[' + dept_nm + ']', DATALENGTH(dept_nm),
       DATALENGTH(LTRIM(RTRIM(dept_nm))), COUNT(*)
  FROM ktws.DIM_MNG_USER
 WHERE DATALENGTH(dept_nm) <> DATALENGTH(LTRIM(RTRIM(dept_nm)))
 GROUP BY dept_nm
UNION ALL
SELECT 'SC', '[' + name + ']', DATALENGTH(name),
       DATALENGTH(LTRIM(RTRIM(name))), COUNT(*)
  FROM ktws.DIM_MNG_USER
 WHERE DATALENGTH(name) <> DATALENGTH(LTRIM(RTRIM(name)))
 GROUP BY name
ORDER BY 구분, 값
"""
df_values = pd.read_sql(sql, conn)
print(f"총 {len(df_values)}종")
df_values

## 5. 핵심 — 필터가 실제로 0건이 되는지 재현

GOLD와 **같은 방식**(LIKE 패딩)으로 필터를 걸어 본다.
필터 없이 세면 27명, `'토요타 동대문'`으로 걸면 0명이 나와야 문제가 확증된다.

마지막 열은 `LTRIM(RTRIM())` 을 넣었을 때 복구되는지 보여준다 — 코드 방어의 효과 확인용.

In [ ]:
TARGET = "토요타 동대문"   # 끝 공백이 있는 전시장

sql = f"""
DECLARE @GroupName NVARCHAR(MAX) = N'토요타 동대문';
DECLARE @Pad NVARCHAR(MAX) = CASE WHEN @GroupName IS NULL THEN NULL
                                  ELSE N',' + @GroupName + N',' END;

SELECT
      (SELECT COUNT(*) FROM ktws.DIM_MNG_USER
        WHERE DATALENGTH(group_name) <> DATALENGTH(LTRIM(RTRIM(group_name)))) AS 공백값_사용자수
    , (SELECT COUNT(*) FROM ktws.DIM_MNG_USER U
        WHERE @Pad LIKE N'%,' + U.group_name + N',%')                          AS 현재_필터_결과
    , (SELECT COUNT(*) FROM ktws.DIM_MNG_USER U
        WHERE @Pad LIKE N'%,' + LTRIM(RTRIM(U.group_name)) + N',%')            AS trim_적용시
"""
df_filter = pd.read_sql(sql, conn)
print(df_filter.to_string(index=False))
df_filter

## 6. 영향 범위 — 필터로 사라지는 인원

공백이 있는 각 값에 대해, 그 이름으로 필터를 걸었을 때 몇 명이 누락되는지 센다.
`누락_인원 > 0` 인 행이 전부 "필터하면 0건이 되는" 대상이다.

In [ ]:
sql = """
WITH bad AS (
    SELECT '전시장' AS 구분, group_name AS 원본, LTRIM(RTRIM(group_name)) AS 정리값
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(group_name) <> DATALENGTH(LTRIM(RTRIM(group_name)))
     GROUP BY group_name
    UNION ALL
    SELECT '팀', dept_nm, LTRIM(RTRIM(dept_nm))
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(dept_nm) <> DATALENGTH(LTRIM(RTRIM(dept_nm)))
     GROUP BY dept_nm
    UNION ALL
    SELECT 'SC', name, LTRIM(RTRIM(name))
      FROM ktws.DIM_MNG_USER
     WHERE DATALENGTH(name) <> DATALENGTH(LTRIM(RTRIM(name)))
     GROUP BY name
)
SELECT b.구분, '[' + b.원본 + ']' AS 저장된_값, b.정리값,
       -- 주의: 여기서도 = 만 쓰면 안 된다. SQL Server 의 = 는 끝 공백을 무시해서
       -- 공백 없는 동명의 값까지 함께 세어 인원이 부풀려진다(실측: 119명 → 243명).
       -- DATALENGTH 까지 같아야 "그 값 자체"다.
       (SELECT COUNT(*) FROM ktws.DIM_MNG_USER U
         WHERE (b.구분 = '전시장' AND U.group_name = b.원본
                AND DATALENGTH(U.group_name) = DATALENGTH(b.원본))
            OR (b.구분 = '팀'     AND U.dept_nm = b.원본
                AND DATALENGTH(U.dept_nm) = DATALENGTH(b.원본))
            OR (b.구분 = 'SC'     AND U.name = b.원본
                AND DATALENGTH(U.name) = DATALENGTH(b.원본))) AS 누락_인원
FROM bad b
ORDER BY 누락_인원 DESC, b.구분
"""
df_impact = pd.read_sql(sql, conn)
print(f"필터 시 누락되는 총 인원: {df_impact['누락_인원'].sum()}명 ({len(df_impact)}종)")
df_impact

## 7. 판단에 필요한 확인

이 노트북으로 확인되는 것:

1. 저장된 값에 끝 공백이 있다 (`DATALENGTH` 로만 검출됨)
2. 그 값으로 LIKE 패딩 필터를 걸면 **0건**이 된다
3. `LTRIM(RTRIM())` 을 넣으면 복구된다

**아직 확인되지 않은 것 — Power BI 쪽 동작.**
BI는 DAX라 문자열 비교 규칙이 다를 수 있다.
BI에서 전시장 필터로 `토요타 동대문`을 선택했을 때

- **데이터가 나오면** → 우리 LIKE 패딩 방식만의 문제 → 코드에서 `LTRIM(RTRIM())` 방어
- **0건이면** → 원천 데이터 문제 → `DIM_MNG_USER` 정리(ETL 포함)가 근본 해결

어느 쪽이냐에 따라 고칠 위치가 달라진다.

In [ ]:
conn.close()
print('연결 종료')